In [ ]:
import matplotlib.pyplot as plt
import sys
# sys.path.append('src')
from src.models import CreativityReward
# from cre_model_utils import CreativityReward
import torch
from PIL import Image
from matplotlib import gridspec

In [ ]:
device = 'cuda'
backbone = 'siglip-gemma3'
crereward = CreativityReward(device=device, 
                 backbone=backbone)
head_path = 'ckpt/siglip-gemma3/reward_head_best.ckpt'
crereward.reward_head.load_state_dict(torch.load(head_path), strict=True)
_ = crereward.eval()

In [ ]:
image1 = Image.open('test_images/chair_0.png')
image2 = Image.open('test_images/chair_1.png')

reward1 = crereward.compute_reward(image1).detach().cpu().numpy().flatten()
reward2 = crereward.compute_reward(image2).detach().cpu().numpy().flatten()



In [ ]:
REWARD_LABELS = ['Geometery', 'Material', 'Texture', 'Overall']


def plot_reward(fname, reward, ax=None, fig=None, figsize=(5, 8), yticks_on=True):
    """Image + 4 reward bars via an internal 2-row GridSpec.

    ax: optional parent SubplotSpec in an outer GridSpec (e.g. outer_gs[0]).
        If None, creates a new figure.
    fig: figure to draw on when ax is a SubplotSpec (defaults to current figure).
    """

    if ax is None:
        fig = plt.figure(figsize=figsize)
        gs = gridspec.GridSpec(2, 1, figure=fig, hspace=0.05, height_ratios=[5, 3])
    else:
        fig = fig if fig is not None else plt.gcf()
        gs = gridspec.GridSpecFromSubplotSpec(
            2, 1, ax, hspace=0.05, height_ratios=[5, 3]
        )

    ax_img = fig.add_subplot(gs[0])
    test_image = Image.open(fname).resize((896, 896)).convert('RGB')
    ax_img.imshow(test_image)
    ax_img.set_xticks([])
    ax_img.set_yticks([])

    ax_bar = fig.add_subplot(gs[1])
    ax_bar.barh([3, 2, 1, 0], reward, height=0.5, color=['C0', 'C1', 'C2', 'C3'], alpha=0.7, edgecolor='k')
    ax_bar.axvline(0, color='k', alpha=0.4, linestyle='--')
    ax_bar.set_xlim(-2, 2)
    if yticks_on:
        ax_bar.set_yticks(range(4), REWARD_LABELS[::-1])
    else:
        ax_bar.set_yticks([])

    return fig

In [ ]:
fig = plt.figure(figsize=(10, 10))
outer_gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.05)
plot_reward("test_images/chair_0.png", reward1, ax=outer_gs[0], fig=fig)
plot_reward("test_images/chair_1.png", reward2, ax=outer_gs[1], fig=fig, yticks_on=False)
plt.show()
